# LightGBM — Qualifying Position Predictor

Hyperparameter tuning via **Optuna** (75 trials, TPE sampler) with native categorical support and early stopping.  
Val years: 2021 / 2022 / 2023 | Test: 2024–2025.

In [ ]:
import pandas as pd
import numpy as np
import joblib, json, os, warnings
warnings.filterwarnings('ignore')

PROC_CSV = r'C:\Users\CLL\OneDrive\Documents\GitHub\F1-Race-Predictor\notebooks\f1_processed.csv'
MDL_DIR  = r'C:\Users\CLL\OneDrive\Documents\GitHub\F1-Race-Predictor\models'
os.makedirs(MDL_DIR, exist_ok=True)

# ── Feature columns (leakage-free — no Q1/Q2/Q3 gaps, no appearance flags) ──
FEATURE_COLS = [
    "reg_disruption_index", "years_since_reg_change",
    "season_stage_ratio", "driver_career_races", "driver_q3_rate_season",
    "driver_circuit_q_pos_hist", "prior_year_q_pos_same_circuit",
    "team_rolling_q_pos_5r", "driver_rolling_race_pos_5r", "teammate_q_gap_season",
    "is_night_race", "is_street_circuit",
    "driver_cum_pts", "team_cum_pts",
    "driver_pts_gap_to_leader", "team_pts_gap_to_leader",
    "driver_q_vs_race_delta_5r",
    "circuit_altitude_m", "circuit_length_km", "num_corners", "num_drs_zones",
    "driver_age", "is_home_race",
    "fp1_gap", "fp2_gap", "fp3_gap",
    "is_wet_qualifying", "track_temp_avg", "air_temp_avg", "humidity_avg", "wind_speed_avg",
    "has_fp_data",          # explicit NaN flag for 2023+ only features
]
TARGET = "GridPosition"
# Rolling-window validation years (train on all prior, validate on this year)
VAL_YEARS  = [2021, 2022, 2023]
TEST_YEARS = [2024, 2025]

## Load & Prepare Data

In [ ]:
df = pd.read_csv(PROC_CSV)

# Derived columns not saved to CSV
df["has_fp_data"] = df["fp1_gap"].notna().astype(int)

# Drop rows with missing target
df = df[df[TARGET].notna()].copy()
df = df.sort_values(["Season","Round"]).reset_index(drop=True)

print(f"Loaded: {len(df)} rows | seasons: {sorted(df['Season'].unique())}")
print(f"Missing values per feature:")
print(df[FEATURE_COLS].isna().sum()[df[FEATURE_COLS].isna().sum() > 0].to_string())

## Encode Categorical Feature

`power_unit` label-encoded and passed as `categorical_feature` to LightGBM for native tree splitting.

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["power_unit_enc"] = le.fit_transform(df["power_unit"].fillna("Unknown"))
joblib.dump(le, os.path.join(MDL_DIR, "lgbm_pu_encoder.joblib"))

XCOLS = FEATURE_COLS + ["power_unit_enc"]
CAT_FEATURES = ["power_unit_enc"]
X = df[XCOLS]
y = df[TARGET].values
seasons = df["Season"].values

## Hyperparameter Tuning (Optuna)

Search space: `n_estimators`, `num_leaves`, `learning_rate`, `subsample`, `colsample_bytree`, `reg_alpha`, `reg_lambda`, `min_child_samples`. Early stopping (50 rounds) inside each fold.

In [ ]:
import optuna, lightgbm as lgb
optuna.logging.set_verbosity(optuna.logging.WARNING)

def lgb_objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int("n_estimators", 300, 1500),
        num_leaves        = trial.suggest_int("num_leaves", 20, 200),
        learning_rate     = trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        subsample         = trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree  = trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha         = trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda        = trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        min_child_samples = trial.suggest_int("min_child_samples", 5, 50),
        random_state=42, verbose=-1,
    )
    maes = []
    for val_yr in VAL_YEARS:
        tr = seasons < val_yr
        va = seasons == val_yr
        m = lgb.LGBMRegressor(**params)
        m.fit(X[tr], y[tr],
              eval_set=[(X[va], y[va])],
              categorical_feature=CAT_FEATURES,
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
        maes.append(np.mean(np.abs(m.predict(X[va]) - y[va])))
    return np.mean(maes)

study_lgb = optuna.create_study(direction="minimize",
                                 study_name="lgb_quali",
                                 sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(lgb_objective, n_trials=75, show_progress_bar=True)

print(f"\nBest CV MAE: {study_lgb.best_value:.4f}")
print("Best params:", study_lgb.best_params)

## Final Model — Train & Evaluate

In [ ]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt

best_p = study_lgb.best_params
best_p.update({"random_state":42,"verbose":-1})

tr_mask = seasons <= 2023
te_mask = np.isin(seasons, TEST_YEARS)

lgb_final = lgb.LGBMRegressor(**best_p)
lgb_final.fit(X[tr_mask], y[tr_mask],
              categorical_feature=CAT_FEATURES,
              callbacks=[lgb.log_evaluation(0)])

preds_test = lgb_final.predict(X[te_mask])
mae  = mean_absolute_error(y[te_mask], preds_test)
rmse = np.sqrt(np.mean((preds_test - y[te_mask])**2))
print(f"Test MAE : {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")

# Feature importance
fi = lgb_final.feature_importances_
fig, axes = plt.subplots(1, 2, figsize=(15, 8))
order = np.argsort(fi)
axes[0].barh([XCOLS[i] for i in order], fi[order], color="#2ecc71")
axes[0].set_title("LightGBM feature importance (gain)")
axes[0].set_xlabel("Importance")

axes[1].scatter(y[te_mask], preds_test - y[te_mask], alpha=0.4, s=12, color="#e74c3c")
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_xlabel("Actual GridPosition")
axes[1].set_ylabel("Residual (pred - actual)")
axes[1].set_title(f"Residuals on 2024-2025  (MAE={mae:.3f})")
plt.tight_layout(); plt.show()

fi_df = pd.DataFrame({"feature": XCOLS, "importance": fi}).sort_values("importance", ascending=False)
print("\nTop 10 features:")
print(fi_df.head(10).to_string(index=False))

## Save Model

In [ ]:
joblib.dump(lgb_final, os.path.join(MDL_DIR, "lgbm_quali.joblib"))
with open(os.path.join(MDL_DIR, "lgbm_best_params.json"), "w") as f:
    json.dump(study_lgb.best_params, f, indent=2)
print("Saved: lgbm_quali.joblib + lgbm_best_params.json")